# PREP_01 - Train /  Test Split

## TFM - Skin Lesion Classification (ISIC 2024 / SLICE-3D)

Vamos a trabajar con un artefacto ya generado:

* **Final preprocessed metadata** (`final_preprocessed_from_raw_<timestamp>.parquet`) — Es el resultado de EDA 01 y 02 y contiene todos los datos que necesitamos para hacer el split.



## Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import StratifiedGroupKFold


from skin_lesion_ai.utils.data_utils import (
    load_metadata_parquet,
    save_metadata_parquet,
)

from skin_lesion_ai.visualisation.eda_plots import (
    set_eda_style,
    clean_axis_labels,
)

# Set the EDA style
set_eda_style()

## Cargar Artefacto

### Load Final Preprocessed Metadata

In [2]:
df_preprocessed = load_metadata_parquet(
    stage="processed",
    filename="final_preprocessed_from_raw",
    timestamp_flag=True,
)

print(f"Final preprocessed metadata: {df_preprocessed.shape}")
df_preprocessed.head()

Final preprocessed metadata: (381280, 17)


,isic_id,patient_id,diagnostic_group,target_biopsy,target_malignant,sex,sex_male,age_approx,anatom_site_general,anatom_site_general_code,anatom_site__anterior_torso,anatom_site__head_neck,anatom_site__lower_extremity,anatom_site__posterior_torso,anatom_site__upper_extremity,clin_size_long_diam_mm,clin_size_long_diam_mm_log1p
0,ISIC_0015670,IP_1235828,benign_non_biopsied,0,<NA>,male,1,60.0,lower extremity,4,0,0,1,0,0,3.04,1.396245
1,ISIC_0015845,IP_8170065,benign_non_biopsied,0,<NA>,male,1,60.0,head/neck,5,0,1,0,0,0,1.10,0.741937
2,ISIC_0015864,IP_6724798,benign_non_biopsied,0,<NA>,male,1,60.0,posterior torso,2,0,0,0,1,0,3.40,1.481605
3,ISIC_0015902,IP_4111386,benign_non_biopsied,0,<NA>,male,1,65.0,anterior torso,1,1,0,0,0,0,3.22,1.439835
4,ISIC_0024200,IP_8313778,benign_non_biopsied,0,<NA>,male,1,55.0,anterior torso,1,1,0,0,0,0,2.73,1.316408


In [3]:
# hechamos un vistazo para asegurarnos de que todo va bien
df_preprocessed.info()

<class 'pandas.DataFrame'>
RangeIndex: 381280 entries, 0 to 381279
Data columns (total 17 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   isic_id                       381280 non-null  str    
 1   patient_id                    381280 non-null  str    
 2   diagnostic_group              381280 non-null  string 
 3   target_biopsy                 381280 non-null  int8   
 4   target_malignant              1013 non-null    Int8   
 5   sex                           381280 non-null  str    
 6   sex_male                      381280 non-null  int8   
 7   age_approx                    381280 non-null  float64
 8   anatom_site_general           381280 non-null  str    
 9   anatom_site_general_code      381280 non-null  int8   
 10  anatom_site__anterior_torso   381280 non-null  int8   
 11  anatom_site__head_neck        381280 non-null  int8   
 12  anatom_site__lower_extremity  381280 non-null  int8   


## Using splitter to split df stratified and by groupal id

Have to use the splitter function from sklearn to be able to split the df using a groupal id.

Got from here: https://stackoverflow.com/questions/56872664/complex-dataset-split-stratifiedgroupshufflesplit

Also there is a issue discusion on how/why use the function StratifiedGroupKFold to do the split: https://github.com/scikit-learn/scikit-learn/issues/12076

Also a bit more detailed explanation on split: https://github.com/scikit-learn/scikit-learn/issues/9193


In [4]:
# Split patients into train and test_val sets segun stackoverflow
# he dado ciertos valores provisionales de cara a cuando construya el script de manera definitiva

random_state = 42
test_size = 0.2
desired = 1.0 / test_size
n_folds = int(np.ceil(desired) ) # c=np.ceil(desired), f=np.floor(desired), c if c/desired < desired /f else f

splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
#splitter = GroupShuffleSplit(test_size=.20, n_splits=1, random_state = 42)

#split = splitter.split(df_preprocessed, groups=df_preprocessed['patient_id'])
split = splitter.split(X=df_preprocessed,y=df_preprocessed['target_biopsy'], groups=df_preprocessed['patient_id'])
train_inds, test_val_inds = next(split)

train_df = df_preprocessed.iloc[train_inds]
test_val_df = df_preprocessed.iloc[test_val_inds]

In [5]:
# Hacemos split para test y val a partir de test_val_df

val_size = 0.5
desired_val = 1.0 / val_size
n_folds_val = int(np.ceil(desired_val) )

# Creamos obj splitter para el split que haremos
splitter_val = StratifiedGroupKFold(n_splits=n_folds_val, shuffle=True, random_state=random_state)

# Creamos el obj split para aplicarlo
split_val = splitter_val.split(X=test_val_df,y=test_val_df['target_biopsy'], groups=test_val_df['patient_id'])

# Aplicamos el split 1 iteracion
test_inds, val_inds = next(split_val)

test_df = test_val_df.iloc[test_inds]
val_df = test_val_df.iloc[val_inds]

In [6]:
# Check train and test sets length
print(f"Train set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")
print(f"Val set: {len(val_df)} samples")

Train set: 305024 samples
Test set: 38125 samples
Val set: 38131 samples


In [7]:
train_df

,isic_id,patient_id,diagnostic_group,target_biopsy,target_malignant,sex,sex_male,age_approx,anatom_site_general,anatom_site_general_code,anatom_site__anterior_torso,anatom_site__head_neck,anatom_site__lower_extremity,anatom_site__posterior_torso,anatom_site__upper_extremity,clin_size_long_diam_mm,clin_size_long_diam_mm_log1p
0,ISIC_0015670,IP_1235828,benign_non_biopsied,0,<NA>,male,1,60.0,lower extremity,4,0,0,1,0,0,3.04,1.396245
1,ISIC_0015845,IP_8170065,benign_non_biopsied,0,<NA>,male,1,60.0,head/neck,5,0,1,0,0,0,1.10,0.741937
2,ISIC_0015864,IP_6724798,benign_non_biopsied,0,<NA>,male,1,60.0,posterior torso,2,0,0,0,1,0,3.40,1.481605
3,ISIC_0015902,IP_4111386,benign_non_biopsied,0,<NA>,male,1,65.0,anterior torso,1,1,0,0,0,0,3.22,1.439835
4,ISIC_0024200,IP_8313778,benign_non_biopsied,0,<NA>,male,1,55.0,anterior torso,1,1,0,0,0,0,2.73,1.316408
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
381273,ISIC_9999883,IP_5024708,benign_non_biopsied,0,<NA>,male,1,85.0,anterior torso,1,1,0,0,0,0,7.58,2.149434
381274,ISIC_9999919,IP_3026867,benign_non_biopsied,0,<NA>,male,1,65.0,anterior torso,1,1,0,0,0,0,9.47,2.348514
381276,ISIC_9999951,IP_5678181,benign_non_biopsied,0,<NA>,male,1,60.0,posterior torso,2,0,0,0,1,0,3.11,1.413423
381277,ISIC_9999960,IP_0076153,benign_non_biopsied,0,<NA>,female,0,65.0,anterior torso,1,1,0,0,0,0,2.05,1.115142


In [8]:
test_df

,isic_id,patient_id,diagnostic_group,target_biopsy,target_malignant,sex,sex_male,age_approx,anatom_site_general,anatom_site_general_code,anatom_site__anterior_torso,anatom_site__head_neck,anatom_site__lower_extremity,anatom_site__posterior_torso,anatom_site__upper_extremity,clin_size_long_diam_mm,clin_size_long_diam_mm_log1p
13,ISIC_0051897,IP_5516884,benign_non_biopsied,0,<NA>,male,1,60.0,upper extremity,3,0,0,0,0,1,4.60,1.722767
22,ISIC_0052109,IP_3927284,benign_non_biopsied,0,<NA>,male,1,55.0,posterior torso,2,0,0,0,1,0,16.70,2.873565
29,ISIC_0052241,IP_3880252,benign_non_biopsied,0,<NA>,male,1,75.0,lower extremity,4,0,0,1,0,0,4.07,1.623341
55,ISIC_0073467,IP_9724369,benign_non_biopsied,0,<NA>,male,1,55.0,posterior torso,2,0,0,0,1,0,3.80,1.568616
59,ISIC_0073642,IP_9588095,benign_non_biopsied,0,<NA>,male,1,45.0,upper extremity,3,0,0,0,0,1,2.56,1.269761
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
381227,ISIC_9998874,IP_0658218,benign_non_biopsied,0,<NA>,male,1,75.0,upper extremity,3,0,0,0,0,1,3.12,1.415853
381240,ISIC_9999106,IP_6501396,benign_non_biopsied,0,<NA>,male,1,50.0,upper extremity,3,0,0,0,0,1,4.70,1.740466
381262,ISIC_9999520,IP_3751225,benign_non_biopsied,0,<NA>,male,1,75.0,upper extremity,3,0,0,0,0,1,4.30,1.667707
381266,ISIC_9999678,IP_9724369,benign_non_biopsied,0,<NA>,male,1,55.0,lower extremity,4,0,0,1,0,0,3.26,1.449269


In [9]:
val_df

,isic_id,patient_id,diagnostic_group,target_biopsy,target_malignant,sex,sex_male,age_approx,anatom_site_general,anatom_site_general_code,anatom_site__anterior_torso,anatom_site__head_neck,anatom_site__lower_extremity,anatom_site__posterior_torso,anatom_site__upper_extremity,clin_size_long_diam_mm,clin_size_long_diam_mm_log1p
23,ISIC_0052122,IP_1117889,benign_non_biopsied,0,<NA>,female,0,55.0,anterior torso,1,1,0,0,0,0,2.60,1.280934
30,ISIC_0052259,IP_6949284,benign_non_biopsied,0,<NA>,male,1,45.0,anterior torso,1,1,0,0,0,0,2.81,1.337629
35,ISIC_0052332,IP_3896746,benign_non_biopsied,0,<NA>,male,1,65.0,lower extremity,4,0,0,1,0,0,2.81,1.337629
47,ISIC_0073261,IP_1117889,benign_non_biopsied,0,<NA>,female,0,55.0,anterior torso,1,1,0,0,0,0,6.19,1.972691
53,ISIC_0073412,IP_4150126,benign_non_biopsied,0,<NA>,male,1,65.0,lower extremity,4,0,0,1,0,0,2.88,1.355835
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
381264,ISIC_9999596,IP_1117889,benign_non_biopsied,0,<NA>,female,0,55.0,upper extremity,3,0,0,0,0,1,4.86,1.768150
381269,ISIC_9999779,IP_1117889,benign_non_biopsied,0,<NA>,female,0,55.0,lower extremity,4,0,0,1,0,0,4.36,1.678964
381270,ISIC_9999817,IP_1117889,benign_non_biopsied,0,<NA>,female,0,55.0,anterior torso,1,1,0,0,0,0,3.02,1.391282
381271,ISIC_9999852,IP_1117889,benign_non_biopsied,0,<NA>,female,0,55.0,posterior torso,2,0,0,0,1,0,3.56,1.517323


Para validar que se ha hecho minimamente correcto el split vamos a comprobar que no haya ningun patient en ambos splits a la vez.

In [28]:
# patient check overlaps

df_overlap1 = pd.merge(train_df, test_df,how="inner", on=["patient_id","patient_id"])
print(f"Train ∩ Test overlap: {len(df_overlap1)} patients")

df_overlap2 = pd.merge(val_df, test_df,how="inner", on=["patient_id","patient_id"])
print(f"Validation ∩ Test overlap: {len(df_overlap2)} patients")

df_overlap3 = pd.merge(val_df, train_df,how="inner", on=["patient_id","patient_id"])
print(f"Validation ∩ Train overlap: {len(df_overlap3)} patients")

if (len(df_overlap1)) + len(df_overlap2) + len(df_overlap3) == 0:
    print("✅ No patient overlap")
else:
    print("❌ Patient overlap detected")


Train ∩ Test overlap: 0 patients
Validation ∩ Test overlap: 0 patients
Validation ∩ Train overlap: 0 patients
✅ No patient overlap


Verificamos que la particion que hacemos la hemos hecho stratificada y por lo tanto mantenemos las proporciones en el split.

In [14]:
# checkear stratificacion de nuestro split

preprocessed_biops = df_preprocessed['target_biopsy'].value_counts()
train_biops = train_df['target_biopsy'].value_counts()
test_biops = test_df['target_biopsy'].value_counts()
val_biops = val_df['target_biopsy'].value_counts()

print(f"Preprocessed_df target_biopsy count:\n {preprocessed_biops} \n")
print(f"Shown as percentatge:\n {100*preprocessed_biops/len(df_preprocessed)}  \n% of the samples.\n" )

print("After our split we have: \n")

print(f"Train_df target_biopsy count:\n {train_biops} \n")
print(f"Shown as percentatge:\n {100*train_biops/len(train_df)} \n% of the samples.\n" )

print(f"Test_df target_biopsy count:\n {test_biops} \n")
print(f"Shown as percentatge:\n {100*test_biops/len(test_df)}  \n% of the samples.\n" )

print(f"Val_df target_biopsy count:\n {val_biops} \n")
print(f"Shown as percentatge:\n {100*val_biops/len(val_df)}  \n% of the samples.\n" )

Preprocessed_df target_biopsy count:
 target_biopsy
0    380267
1      1013
Name: count, dtype: int64 

Shown as percentatge:
 target_biopsy
0    99.734316
1     0.265684
Name: count, dtype: float64  
% of the samples.

After our split we have: 

Train_df target_biopsy count:
 target_biopsy
0    304212
1       812
Name: count, dtype: int64 

Shown as percentatge:
 target_biopsy
0    99.733791
1     0.266209
Name: count, dtype: float64 
% of the samples.

Test_df target_biopsy count:
 target_biopsy
0    38025
1      100
Name: count, dtype: int64 

Shown as percentatge:
 target_biopsy
0    99.737705
1     0.262295
Name: count, dtype: float64  
% of the samples.

Val_df target_biopsy count:
 target_biopsy
0    38030
1      101
Name: count, dtype: int64 

Shown as percentatge:
 target_biopsy
0    99.735124
1     0.264876
Name: count, dtype: float64  
% of the samples.



Comprobemos la coherencia del numero de lesiones 

In [24]:
# numero de lesiones en cada split y porcentaje del original (preprocessed)
print(f"Preprocessed set:\n {df_preprocessed['isic_id'].nunique()} lesions\n")

print(f"Train set:\n {train_df['isic_id'].nunique()} lesions")
print(f" {100*train_df['isic_id'].nunique()/df_preprocessed['isic_id'].nunique()} % \nof preprocessed set\n")

print(f"Test set:\n {test_df['isic_id'].nunique()} lesions")
print(f" {100*test_df['isic_id'].nunique()/df_preprocessed['isic_id'].nunique()} % \nof preprocessed set\n")

print(f"Validation set:\n {val_df['isic_id'].nunique()} lesions")
print(f" {100*val_df['isic_id'].nunique()/df_preprocessed['isic_id'].nunique()} % \nof preprocessed set\n")

# se cumple igualdad de train + test + val = preprocessed
print(f"Total lesions in splits: {train_df['isic_id'].nunique() + test_df['isic_id'].nunique() + val_df['isic_id'].nunique()} lesions")
print(f"Preprocessed lesions: {df_preprocessed['isic_id'].nunique()} lesions")

if train_df['isic_id'].nunique() + test_df['isic_id'].nunique() + val_df['isic_id'].nunique() == df_preprocessed['isic_id'].nunique():
    print("✅ Train + Test + Val = Preprocessed")
else:
    print("❌ Train + Test + Val != Preprocessed")

Preprocessed set:
 381280 lesions

Train set:
 305024 lesions
 80.0 % 
of preprocessed set

Test set:
 38125 lesions
 9.999213176668066 % 
of preprocessed set

Validation set:
 38131 lesions
 10.000786823331934 % 
of preprocessed set

Total lesions in splits: 381280 lesions
Preprocessed lesions: 381280 lesions
✅ Train + Test + Val = Preprocessed


Comprobemos coherencia del numero de pacientes

In [26]:
# numero de pacientes en cada split
print(f"Preprocessed set:\n {df_preprocessed['patient_id'].nunique()} patients\n")

print(f"Train set:\n {train_df['patient_id'].nunique()} patients")
print(f" {100*train_df['patient_id'].nunique()/df_preprocessed['patient_id'].nunique()} % \nof preprocessed set\n")

print(f"Test set:\n {test_df['patient_id'].nunique()} patients")
print(f" {100*test_df['patient_id'].nunique()/df_preprocessed['patient_id'].nunique()} % \nof preprocessed set\n")

print(f"Validation set:\n {val_df['patient_id'].nunique()} patients")
print(f" {100*val_df['patient_id'].nunique()/df_preprocessed['patient_id'].nunique()} % \nof preprocessed set\n")

# Se cumple igualdad de train + test + val = preprocessed
print(f"Total patients in splits: {train_df['patient_id'].nunique() + test_df['patient_id'].nunique() + val_df['patient_id'].nunique()} patients")
print(f"Preprocessed patients: {df_preprocessed['patient_id'].nunique()} patients")

if train_df['patient_id'].nunique() + test_df['patient_id'].nunique() + val_df['patient_id'].nunique() == df_preprocessed['patient_id'].nunique():
    print("✅ Train + Test + Val = Preprocessed")
else:
    print("❌ Train + Test + Val != Preprocessed")

Preprocessed set:
 977 patients

Train set:
 784 patients
 80.24564994882293 % 
of preprocessed set

Test set:
 99 patients
 10.133060388945752 % 
of preprocessed set

Validation set:
 94 patients
 9.62128966223132 % 
of preprocessed set

Total patients in splits: 977 patients
Preprocessed patients: 977 patients
✅ Train + Test + Val = Preprocessed


Ahora que tenemos hecho el split debemos salvar los dataframes que hemos obtenido.

In [12]:
#saves


train_path = save_metadata_parquet(
    train_df,
    stage="processed",
    name="train",
    timestamp=True,
)



val_path = save_metadata_parquet(
    val_df,
    stage="processed",
    name="validation",
    timestamp=True,
)


test_path = save_metadata_parquet(
    test_df,
    stage="processed",
    name="test",
    timestamp=True,
)

print(f"Train saved to: {train_path}")
print(f"Validation saved to: {val_path}")
print(f"Test saved to: {test_path}")

Train saved to: C:\Users\Krop\Desktop\provoMasterJul26\my-image-classifier\data\processed\metadata\train_20260711_212056.parquet
Validation saved to: C:\Users\Krop\Desktop\provoMasterJul26\my-image-classifier\data\processed\metadata\validation_20260711_212056.parquet
Test saved to: C:\Users\Krop\Desktop\provoMasterJul26\my-image-classifier\data\processed\metadata\test_20260711_212056.parquet
